# 1. Libraries

Import the libraries used for tabular data, vector search, and semantic embeddings. **Outcome:** the tools needed for the retrieval experiments are available.

In [29]:
import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from ranx import Qrels, Run, evaluate  


# 2. Dataset loading and inspection

Load the IMDb dataset and inspect its structure, data types, and missing values. **Observation:** the movie overviews and core identifying fields are complete; other metadata fields have missing values.

In [30]:
df = pd.read_csv("data/imdb_movies.csv")

print("Dimensões:", df.shape)

display(df.head())

Dimensões: (999, 16)


,Unnamed: 0,Series_Title,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Overview,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross
0,1,The Godfather,1972,A,175 min,"Crime, Drama",9.2,An organized crime dynasty's aging patriarch t...,100.0,Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,1620367,"134,966,411"
1,2,The Dark Knight,2008,UA,152 min,"Action, Crime, Drama",9.0,When the menace known as the Joker wreaks havo...,84.0,Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,2303232,"534,858,444"
2,3,The Godfather: Part II,1974,A,202 min,"Crime, Drama",9.0,The early life and career of Vito Corleone in ...,90.0,Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton,1129952,"57,300,000"
3,4,12 Angry Men,1957,U,96 min,"Crime, Drama",9.0,A jury holdout attempts to prevent a miscarria...,96.0,Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler,689845,"4,360,000"
4,5,The Lord of the Rings: The Return of the King,2003,U,201 min,"Action, Adventure, Drama",8.9,Gandalf and Aragorn lead the World of Men agai...,94.0,Peter Jackson,Elijah Wood,Viggo Mortensen,Ian McKellen,Orlando Bloom,1642758,"377,845,905"


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 999 entries, 0 to 998
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Unnamed: 0     999 non-null    int64  
 1   Series_Title   999 non-null    str    
 2   Released_Year  999 non-null    str    
 3   Certificate    898 non-null    str    
 4   Runtime        999 non-null    str    
 5   Genre          999 non-null    str    
 6   IMDB_Rating    999 non-null    float64
 7   Overview       999 non-null    str    
 8   Meta_score     842 non-null    float64
 9   Director       999 non-null    str    
 10  Star1          999 non-null    str    
 11  Star2          999 non-null    str    
 12  Star3          999 non-null    str    
 13  Star4          999 non-null    str    
 14  No_of_Votes    999 non-null    int64  
 15  Gross          830 non-null    str    
dtypes: float64(2), int64(2), str(12)
memory usage: 125.0 KB


In [4]:
df.isnull().sum()

Unnamed: 0         0
Series_Title       0
Released_Year      0
Certificate      101
Runtime            0
Genre              0
IMDB_Rating        0
Overview           0
Meta_score       157
Director           0
Star1              0
Star2              0
Star3              0
Star4              0
No_of_Votes        0
Gross            169
dtype: int64

**Inspection takeaway:** Missing values occur in some metadata fields, but the overview and selected identifying fields are complete. No rows are removed at this stage.

# 3. Column selection

Select the fields to represent each movie, including cast information. This preserves the original dataset and creates a working copy.

In [5]:
columns = [
    "Series_Title",
    "Released_Year",
    "Genre",
    "Director",
    "Star1",
    "Star2",
    "Star3",
    "Star4",
    "Overview",
    "IMDB_Rating"
]

movies = df[columns].copy()

# 4. Document construction

Combine the selected metadata and overview into one text document per movie. **Outcome:** each movie has a textual representation for embedding.

In [31]:
def create_document(row):
    return (
        f"Title: {row['Series_Title']}\n"
        f"Year: {row['Released_Year']}\n"
        f"Genre: {row['Genre']}\n"
        f"Director: {row['Director']}\n"
        f"Star1: {row['Star1']}\n"
        f"Star2: {row['Star2']}\n"
        f"Star3: {row['Star3']}\n"
        f"Star4: {row['Star4']}\n"
        f"IMDb Rating: {row['IMDB_Rating']}\n"
        f"Overview: {row['Overview']}"
    )


movies["document"] = movies.apply(create_document, axis=1)

In [7]:
print(movies["document"].iloc[0])

Title: The Godfather
Year: 1972
Genre: Crime, Drama
Director: Francis Ford Coppola
Star1: Marlon Brando
Star2: Al Pacino
Star3: James Caan
Star4: Diane Keaton
IMDb Rating: 9.2
Overview: An organized crime dynasty's aging patriarch transfers control of his clandestine empire to his reluctant son.


In [8]:
print("Total documents:", len(movies))
print("Empty documents:", movies["document"].isna().sum())

Total documents: 999
Empty documents: 0


**Document preparation takeaway:** The documents retain cast, director, genre, rating, and overview. They are ready for semantic embedding.

# 5. Load the embedding model

Use the pretrained `all-MiniLM-L6-v2` model to encode text into 384-dimensional vectors.

In [32]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 204.08it/s]


# 6. Generate document embeddings

Encode the complete movie documents and apply L2 normalization. **Outcome:** each movie is represented by a unit-length vector.

In [33]:
documents = movies["document"].tolist()

embeddings = model.encode(
    documents,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Shape dos embeddings:", embeddings.shape)

Batches: 100%|██████████| 32/32 [00:28<00:00,  1.11it/s]

Shape dos embeddings: (999, 384)


In [11]:
print(embeddings[0][:10])

[-0.10328615 -0.043273   -0.06441569 -0.01188476 -0.07674628  0.01037306
 -0.00513457 -0.01141561  0.02762846 -0.03084671]


**Embedding takeaway:** The output should have shape `(999, 384)`. Individual dimensions are not directly interpretable.

# 7. Build the FAISS index

Store normalized embeddings in an `IndexFlatIP` index. Because the vectors have unit length, inner-product scores equal cosine similarity (up to numerical precision).

In [34]:
embeddings = np.asarray(embeddings, dtype="float32")

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("Total de vetores indexados:", index.ntotal)

Total de vetores indexados: 999


**Index takeaway:** The index should contain 999 normalized vectors. Scores are cosine similarities, not probabilities of relevance.

# 8. Semantic search function

Encode a query with the same model and normalization, then retrieve the highest-scoring movie documents.

In [35]:
def search_movies(query, top_k=5):

    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, top_k)

    results = movies.iloc[indices[0]].copy()
    results["similarity"] = scores[0]

    return results[
        ["Series_Title", "Genre", "Overview", "similarity"]
    ]

# 9. Initial retrieval test

Inspect results for a query about organized crime families. **Observation from the initial run:** *The Godfather* appeared at rank 10, indicating that semantic similarity does not guarantee an ideal relevance ranking.

In [14]:
query = "Movies about organized crime families"

results = search_movies(query, top_k=20)

display(results)

,Series_Title,Genre,Overview,similarity
790,Road to Perdition,"Crime, Drama, Thriller","A mob enforcer's son witnesses a murder, forci...",0.500638
164,Casino,"Crime, Drama","A tale of greed, deception, money, power, and ...",0.498396
978,Lethal Weapon,"Action, Crime, Thriller",Two newly paired cops who are complete opposit...,0.484436
160,"Lock, Stock and Two Smoking Barrels","Action, Comedy, Crime",A botched card game in London triggers four fr...,0.484253
307,White Heat,"Action, Crime, Drama",A psychopathic criminal with a mother complex ...,0.483981
487,District 9,"Action, Sci-Fi, Thriller",Violence ensues after an extraterrestrial race...,0.482105
14,Goodfellas,"Biography, Crime, Drama",The story of Henry Hill and his life in the mo...,0.481330
950,Minority Report,"Action, Crime, Mystery",In a future where a special police unit is abl...,0.480426
5,Pulp Fiction,"Crime, Drama","The lives of two mob hitmen, a boxer, a gangst...",0.478042
0,The Godfather,"Crime, Drama",An organized crime dynasty's aging patriarch t...,0.472166


**Initial test takeaway:** Relevant crime films appear, but some off-topic results are also retrieved. This motivates systematic evaluation.

# 10. Overview-only retrieval experiment

Create a second index using only movie overviews. The embedding model and FAISS index type remain unchanged, isolating the effect of document representation. **Observation from the initial run:** *The Godfather* ranked 18th for this example; one query is insufficient to select the better strategy.

In [15]:
overview_embeddings = model.encode(
    movies["Overview"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
).astype("float32")

Batches: 100%|██████████| 32/32 [00:12<00:00,  2.50it/s]


In [16]:
overview_index = faiss.IndexFlatIP(
    overview_embeddings.shape[1]
)

overview_index.add(overview_embeddings)

In [17]:
query = "Movies about organized crime families"

query_embedding = model.encode(
    [query],
    normalize_embeddings=True
).astype("float32")

scores, indices = overview_index.search(query_embedding, 20)

overview_results = movies.iloc[indices[0]][
    ["Series_Title", "Overview"]
].copy()

overview_results["similarity"] = scores[0]

display(overview_results)

,Series_Title,Overview,similarity
430,Per un pugno di dollari,A wandering gunfighter plays two rival familie...,0.503125
363,Sin City,A movie that explores the dark and miserable t...,0.499590
90,Miracle in cell NO.7,A story of love between a mentally-ill father ...,0.472946
900,End of Watch,"Shot documentary-style, this film follows the ...",0.460586
176,Sholay,After his family is murdered by a notorious an...,0.456599
5,Pulp Fiction,"The lives of two mob hitmen, a boxer, a gangst...",0.452530
161,L.A. Confidential,"As corruption grows in 1950s Los Angeles, thre...",0.451694
124,M - Eine Stadt sucht einen Mörder,When the police in a German city are unable to...,0.447814
307,White Heat,A psychopathic criminal with a mother complex ...,0.440914
790,Road to Perdition,"A mob enforcer's son witnesses a murder, forci...",0.439556


**Overview-only takeaway:** The two rankings differ. We will compare them across multiple queries instead of drawing a conclusion from a single example.

# 11. Preliminary three-query evaluation

Use a small, manually selected relevance set for an exploratory comparison. **Limitation:** these labels are incomplete, so the results are illustrative rather than definitive.

In [51]:
evaluation_queries = [
    {
        "query": "Movies about organized crime families",
        "relevant_movies": [
            "The Godfather",
            "Goodfellas",
            "Casino",
            "Road to Perdition"
        ]
    },
    {
        "query": "Movies involving space exploration",
        "relevant_movies": [
            "Interstellar",
            "2001: A Space Odyssey"
        ]
    },
    {
        "query": "Movies about World War II",
        "relevant_movies": [
            "Schindler's List",
            "Saving Private Ryan"
        ]
    }
]

In [52]:
for item in evaluation_queries:
    for title in item["relevant_movies"]:
        exists = title in movies["Series_Title"].values
        print(f"{title}: {exists}")

The Godfather: True
Goodfellas: True
Casino: True
Road to Perdition: True
Interstellar: True
2001: A Space Odyssey: True
Schindler's List: True
Saving Private Ryan: True


In [53]:
# Use ranx instead of implementing Recall@K manually.
def recall_at_k(retrieved_movies, relevant_movies):
    qrels = Qrels({"q": {title: 1 for title in relevant_movies}})
    # Descending rank scores preserve the retrieved order.
    run = Run({"q": {title: float(len(retrieved_movies) - rank)
                     for rank, title in enumerate(retrieved_movies)}})
    return float(evaluate(qrels, run, f"recall@{len(retrieved_movies)}"))


In [54]:
def evaluate_retrieval(index, evaluation_queries, k=10):
    # Preserve the original output structure; delegate metric computation to ranx.
    results = []
    for item in evaluation_queries:
        query = item["query"]
        query_embedding = model.encode([query], normalize_embeddings=True).astype("float32")
        scores, indices = index.search(query_embedding, k)
        retrieved_movies = movies.iloc[indices[0]]["Series_Title"].tolist()
        recall = recall_at_k(retrieved_movies, item["relevant_movies"])
        results.append({"query": query, "recall": recall})
    return pd.DataFrame(results)


In [55]:
results_full = evaluate_retrieval(
    index,
    evaluation_queries,
    k=10
)

results_overview = evaluate_retrieval(
    overview_index,
    evaluation_queries,
    k=10
)

comparison = pd.DataFrame({
    "Query": results_full["query"],
    "Recall@10 (Full)": results_full["recall"],
    "Recall@10 (Overview)": results_overview["recall"]
})

display(comparison)

,Query,Recall@10 (Full),Recall@10 (Overview)
0,Movies about organized crime families,1.0,0.25
1,Movies involving space exploration,0.5,0.50
2,Movies about World War II,0.0,0.00


In [56]:
print(
    "Full documents:",
    comparison["Recall@10 (Full)"].mean()
)

print(
    "Overview only:",
    comparison["Recall@10 (Overview)"].mean()
)

Full documents: 0.5
Overview only: 0.25


**Preliminary evaluation limitation:** The three-query relevance lists are incomplete and should not be used as final performance estimates.

# 12. Evaluation using the 29-query reference set

Load the larger reference set and compare both retrieval strategies using `ranx` for standard information-retrieval metrics. Metadata-derived labels are distinct from provisional semantic labels; do not treat provisional labels as verified ground truth.

In [57]:
import json

with open(
    "data/movie_rag_evaluation_queries.json",
    "r",
    encoding="utf-8"
) as f:
    evaluation_queries = json.load(f)

print("Total queries:", len(evaluation_queries))

print(evaluation_queries[0])

Total queries: 29
{'query_id': 'Q01', 'query': 'Which movies were directed by Christopher Nolan?', 'category': 'director', 'relevant_movies': ['The Dark Knight', 'Inception', 'Interstellar', 'The Prestige', 'The Dark Knight Rises', 'Memento', 'Batman Begins', 'Dunkirk'], 'number_relevant': 8, 'label_status': 'verified_from_metadata', 'notes': ''}


In [25]:
# Recall@K is computed by ranx through the function defined in Section 11.
# Reuse that function rather than redefining the metric.


In [58]:
def evaluate_retrieval(index, evaluation_queries, k=10):
    # Keep the same inputs and result columns as the original 29-query evaluation.
    results = []
    for item in evaluation_queries:
        query = item["query"]
        query_embedding = model.encode([query], normalize_embeddings=True).astype("float32")
        scores, indices = index.search(query_embedding, k)
        retrieved_movies = movies.iloc[indices[0]]["Series_Title"].tolist()
        recall = recall_at_k(retrieved_movies, item["relevant_movies"])
        results.append({
            "query_id": item["query_id"],
            "category": item["category"],
            "label_status": item["label_status"],
            "query": query,
            f"Recall@{k}": recall
        })
    return pd.DataFrame(results)


In [59]:
results = []

for k in [5, 10]:

    for name, faiss_index in [
        ("Full Document", index),
        ("Overview Only", overview_index)
    ]:

        evaluation = evaluate_retrieval(
            faiss_index,
            evaluation_queries,
            k=k
        )

        evaluation["strategy"] = name
        results.append(evaluation)

evaluation_df = pd.concat(results, ignore_index=True)

evaluation_df = (
    evaluation_df
    .groupby(
        ["query_id", "category", "label_status", "query", "strategy"],
        as_index=False
    )[["Recall@5", "Recall@10"]]
    .first()
)

display(evaluation_df.head())

,query_id,category,label_status,query,strategy,Recall@5,Recall@10
0,Q01,director,verified_from_metadata,Which movies were directed by Christopher Nolan?,Full Document,0.625000,0.875000
1,Q01,director,verified_from_metadata,Which movies were directed by Christopher Nolan?,Overview Only,0.000000,0.000000
2,Q02,director,verified_from_metadata,Which movies were directed by Francis Ford Cop...,Full Document,0.800000,1.000000
3,Q02,director,verified_from_metadata,Which movies were directed by Francis Ford Cop...,Overview Only,0.000000,0.000000
4,Q03,director,verified_from_metadata,Which movies were directed by Steven Spielberg?,Full Document,0.384615,0.692308


In [47]:
summary = (
    evaluation_df
    .melt(
        id_vars=["strategy", "label_status"],
        value_vars=["Recall@5", "Recall@10"],
        var_name="metric",
        value_name="score"
    )
    .dropna(subset=["score"])
    .groupby(["strategy", "label_status", "metric"])["score"]
    .mean()
    .unstack("metric")
)

display(summary)

metric                                     Recall@10  Recall@5
strategy      label_status                                    
Full Document provisional_requires_review   0.249377  0.165337
              verified_from_metadata        0.339824  0.221241
Overview Only provisional_requires_review   0.464242  0.232054
              verified_from_metadata        0.028697  0.014613

**Evaluation takeaway:** Compare `Recall@5` and `Recall@10` separately for metadata-verified and provisional labels. Scores from provisional labels remain exploratory; review those labels before drawing final conclusions.

# TESTEEE com tds as metricas

In [60]:
def evaluate_retrieval(index, evaluation_queries, k=10):

    results = []

    for item in evaluation_queries:

        query_id = item["query_id"]
        query = item["query"]

        query_embedding = model.encode(
            [query],
            normalize_embeddings=True
        ).astype("float32")

        scores, indices = index.search(query_embedding, k)

        retrieved_movies = movies.iloc[
            indices[0]
        ]["Series_Title"].tolist()

        qrels = Qrels({
            query_id: {
                title: 1
                for title in item["relevant_movies"]
            }
        })

        run = Run({
            query_id: {
                title: float(score)
                for title, score in zip(
                    retrieved_movies, scores[0]
                )
            }
        })

        metrics = evaluate(
            qrels,
            run,
            [
                "recall@5",
                "recall@10",
                "precision@5",
                "precision@10",
                "mrr@5",
                "mrr@10",
                "ndcg@5",
                "ndcg@10"
            ]
        )

        results.append({
            "query_id": query_id,
            "category": item["category"],
            "label_status": item["label_status"],
            "query": query,
            **metrics
        })

    return pd.DataFrame(results)

In [61]:
results = []

for name, faiss_index in [
    ("Full Document", index),
    ("Overview Only", overview_index)
]:

    evaluation = evaluate_retrieval(
        faiss_index,
        evaluation_queries,
        k=10
    )

    evaluation["strategy"] = name

    results.append(evaluation)

evaluation_df = pd.concat(
    results,
    ignore_index=True
)

display(evaluation_df.head())

,query_id,category,label_status,query,recall@5,recall@10,precision@5,precision@10,mrr@5,mrr@10,ndcg@5,ndcg@10,strategy
0,Q01,director,verified_from_metadata,Which movies were directed by Christopher Nolan?,0.625000,0.875000,1.0,0.7,1.0,1.0,1.000000,0.915686,Full Document
1,Q02,director,verified_from_metadata,Which movies were directed by Francis Ford Cop...,0.800000,1.000000,0.8,0.5,1.0,1.0,0.830420,0.943473,Full Document
2,Q03,director,verified_from_metadata,Which movies were directed by Steven Spielberg?,0.384615,0.692308,1.0,0.9,1.0,1.0,1.000000,0.936379,Full Document
3,Q04,director,verified_from_metadata,Which movies were directed by Martin Scorsese?,0.200000,0.300000,0.4,0.3,1.0,1.0,0.485229,0.393278,Full Document
4,Q05,director,verified_from_metadata,Which movies were directed by Quentin Tarantino?,0.500000,0.625000,0.8,0.5,1.0,1.0,0.868795,0.727734,Full Document


In [62]:
summary = (
    evaluation_df
    .groupby(["label_status", "strategy"])
    .mean(numeric_only=True)
    .round(4)
)

display(summary)

recall@5  recall@10  precision@5  \
label_status                strategy                                          
provisional_requires_review Full Document    0.1653     0.2494       0.5000   
                            Overview Only    0.2321     0.4642       0.6667   
verified_from_metadata      Full Document    0.2212     0.3398       0.6696   
                            Overview Only    0.0146     0.0287       0.1565   

                                           precision@10   mrr@5  mrr@10  \
label_status                strategy                                      
provisional_requires_review Full Document        0.4500  0.6250  0.6250   
                            Overview Only        0.5333  0.7500  0.7778   
verified_from_metadata      Full Document        0.5652  0.7971  0.8106   
                            Overview Only        0.1261  0.2754  0.2850   

                                           ndcg@5  ndcg@10  
label_status                strategy                        
provisional_requires_review Full Document  0.5192   0.5099  
                            Overview Only  0.6825   0.6628  
verified_from_metadata      Full Document  0.6769   0.6270  
                            Overview Only  0.1608   0.1377